<h1>Imports</h1>

In [1]:
import torch
import torch.nn as nn
import h5py
import os

from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import MessagePassing, knn_graph
from torch_scatter import scatter

e:\anaconda3\envs\torch311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


<h1>Graph_Utils</h1>

In [2]:
def build_edges(positions, k=16):
    return knn_graph(positions, k=k, loop=False)

<h1>Datasets</h1>

In [3]:
class ProteinGraphDataset(torch.utils.data.Dataset):
    def __init__(self, hdf5_path, window_size=15):
        self.window_size = window_size
        self.file = h5py.File(hdf5_path, 'r')
        self.data = self.file['coordinates']

    def __len__(self):
        return len(self.data) - self.window_size

    def __getitem__(self, idx):
        sequence_data = self.data[idx: idx + self.window_size]
        target_pos = self.data[idx + self.window_size]

        seq = torch.from_numpy(sequence_data).float()

        node_features = seq.permute(1, 0, 2).reshape(seq.shape[1], -1)
        current_pos = seq[-1]

        edge_index = build_edges(current_pos)

        y_displacement = torch.from_numpy(target_pos).float() - current_pos

        return Data(
            x=node_features,
            edge_index=edge_index,
            pos=current_pos,
            y=y_displacement
        )

<h1>Model</h1>

In [4]:
class EGNNLayer(MessagePassing):
    def __init__(self, in_channels, out_channels):
        super().__init__(aggr='add')

        self.edge_mlp = nn.Sequential(
            nn.Linear(in_channels * 2 + 1, out_channels),
            nn.SiLU(),
            nn.Linear(out_channels, out_channels)
        )

        self.node_mlp = nn.Sequential(
            nn.Linear(out_channels, out_channels),
            nn.SiLU()
        )

        if in_channels != out_channels:
            self.residual_proj = nn.Linear(in_channels, out_channels)
        else:
            self.residual_proj = nn.Identity()

    def forward(self, x, pos, edge_index):
        row, col = edge_index

        diff = pos[row] - pos[col]
        dist = (diff ** 2).sum(dim=-1, keepdim=True)

        edge_feat = torch.cat([x[row], x[col], dist], dim=-1)

        messages = self.edge_mlp(edge_feat)
        agg = scatter(messages, col, dim=0, dim_size=x.size(0))

        x_res = self.residual_proj(x)
        x = x_res + self.node_mlp(agg)

        return x


class ProteinEGNN(nn.Module):
    def __init__(self, in_channels=45, hidden=64, out_channels=3):
        super().__init__()

        self.layer1 = EGNNLayer(in_channels, hidden)
        self.layer2 = EGNNLayer(hidden, hidden)

        self.out_mlp = nn.Sequential(
            nn.Linear(hidden, hidden),
            nn.SiLU(),
            nn.Linear(hidden, out_channels)
        )

    def forward(self, data):
        x, pos, edge_index = data.x, data.pos, data.edge_index

        x = self.layer1(x, pos, edge_index)
        x = self.layer2(x, pos, edge_index)

        return self.out_mlp(x)

<h1>Training</h1>

In [6]:
# 🔹 Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Training on:", device)

# 🔹 Dataset & Loader
dataset = ProteinGraphDataset("normalized_trajectory.hdf5", window_size=15)
loader = DataLoader(dataset, batch_size=2, shuffle=True)

# 🔹 Model, Optimizer, Loss
model = ProteinEGNN().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = torch.nn.MSELoss()

# 🔹 Checkpoint config
checkpoint_path = "checkpoint.pth"
start_epoch = 0
num_epochs = 20

# 🔁 Resume if checkpoint exists
if os.path.exists(checkpoint_path):
    print("🔄 Loading checkpoint...")
    checkpoint = torch.load(checkpoint_path, map_location=device)

    model.load_state_dict(checkpoint["model_state"])
    optimizer.load_state_dict(checkpoint["optimizer_state"])
    start_epoch = checkpoint["epoch"] + 1

    print(f"✅ Resumed from epoch {start_epoch}")

# 🔥 Training Loop
for epoch in range(start_epoch, num_epochs):
    model.train()
    total_loss = 0

    for i, batch in enumerate(loader):
        batch = batch.to(device)

        pred = model(batch)
        loss = loss_fn(pred, batch.y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        # 🔁 Batch logging every 200 batches
        if i % 200 == 0:
            print(f"Epoch {epoch+1}, Batch {i}, Loss: {loss.item():.4f}")

    print(f"\n✅ Epoch {epoch+1} DONE | Total Loss: {total_loss:.4f}\n")

    # 💾 Save every 2 epochs
    if (epoch + 1) % 2 == 0:
        torch.save({
            "epoch": epoch,
            "model_state": model.state_dict(),
            "optimizer_state": optimizer.state_dict()
        }, checkpoint_path)

        print(f"💾 Checkpoint saved at epoch {epoch+1}")

Training on: cuda
🔄 Loading checkpoint...
✅ Resumed from epoch 4


C:\Users\ASUS\AppData\Local\Temp\ipykernel_33204\2787074520.py:22: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(checkpoint_path, map_location=device

Epoch 5, Batch 0, Loss: 0.0013
Epoch 5, Batch 200, Loss: 0.0013


KeyboardInterrupt: 